In [1]:
import os
import zipfile
import xml.etree.ElementTree as ET
from io import BytesIO
import csv

def parse_all_genotox_to_csv(root_dir, output_csv="genotox_full.csv"):

    rows = []

    def iter_i6d(z):
        for fname in z.namelist():
            if fname.endswith(".i6d"):
                with z.open(fname) as f:
                    yield fname, f.read()

    def get_ref(z):
        compound_name = cas_number = smiles = None
        for fname, content in iter_i6d(z):
            try:
                root = ET.parse(BytesIO(content)).getroot()
            except Exception:
                continue
            for elem in root.iter():
                if elem.tag.split("}")[-1] != "REFERENCE_SUBSTANCE":
                    continue
                for sub in elem.iter():
                    t = sub.tag.split("}")[-1]
                    if t == "IupacName":        compound_name = sub.text
                    if t == "CASNumber":        cas_number    = sub.text
                    if t == "SmilesNotation":   smiles        = sub.text
                if compound_name or cas_number:
                    break
            if compound_name or cas_number:
                break
        return compound_name, cas_number, smiles

    def gv(node, field):
        """node의 직접 자식에서 field 태그를 찾아 value 반환"""
        for e in node:
            if e.tag.split("}")[-1] == field:
                if field == "KeyResult":
                    return e.text
                for s in e.iter():
                    if s.tag.split("}")[-1] == "value":
                        return s.text
        return None

    def get_direct_value(elem):
        """elem 바로 아래 value 태그 반환"""
        for s in elem:
            if s.tag.split("}")[-1] == "value":
                return s.text
        return elem.text

    def process_zip(z, src_fname):
        compound_name, cas_number, smiles = get_ref(z)

        for fname, content in iter_i6d(z):
            try:
                root = ET.parse(BytesIO(content)).getroot()
            except Exception:
                continue

            for study in root.iter():
                if study.tag.split("}")[-1] != "ENDPOINT_STUDY_RECORD.GeneticToxicityVitro":
                    continue

                # ── MaterialsAndMethods ──────────────────────────
                guidelines      = []
                glp_compliance  = None
                type_of_assay   = None
                test_material   = None

                for section in study:
                    if section.tag.split("}")[-1] != "MaterialsAndMethods":
                        continue

                    for child in section:
                        child_tag = child.tag.split("}")[-1]

                        if child_tag == "Guideline":
                            for entry in child:
                                guidelines.append({
                                    "Qualifier": gv(entry, "Qualifier"),
                                    "Guideline": gv(entry, "Guideline"),
                                    "Deviation": gv(entry, "Deviation"),
                                })

                        elif child_tag == "GLPComplianceStatement":
                            glp_compliance = get_direct_value(child)

                        elif child_tag == "TypeOfAssay":
                            type_of_assay = get_direct_value(child)

                        elif child_tag == "TestMaterials":
                            for sub in child.iter():
                                if sub.tag.split("}")[-1] == "TestMaterialInformation":
                                    test_material = sub.text
                                    break

                # 가이드라인 합치기
                # Qualifier: 중복 제거 후 열거
                # Deviation: 중복 제거 후 열거
                # Guideline: 모두 열거
                qualifiers = list(dict.fromkeys(
                    g["Qualifier"] for g in guidelines if g["Qualifier"]
                ))
                deviations = list(dict.fromkeys(
                    g["Deviation"] for g in guidelines if g["Deviation"]
                ))
                guideline_vals = [g["Guideline"] for g in guidelines if g["Guideline"]]

                qualifier_str  = " | ".join(qualifiers)
                deviation_str  = " | ".join(deviations)
                guideline_str  = " | ".join(guideline_vals)

                # ── ResultsAndDiscussion ─────────────────────────
                for section in study:
                    if section.tag.split("}")[-1] != "ResultsAndDiscussion":
                        continue

                    for tr in section:
                        if tr.tag.split("}")[-1] != "TestRs":
                            continue

                        for entry in tr:
                            rows.append({
                                "파일명"                        : src_fname,
                                "Compound name (IUPAC)"        : compound_name,
                                "CAS number"                   : cas_number,
                                "SMILES"                       : smiles,
                                "Qualifier"                    : qualifier_str,
                                "Guideline"                    : guideline_str,
                                "Deviation"                    : deviation_str,
                                "GLP compliance"               : glp_compliance,
                                "Type of assay"                : type_of_assay,
                                "Test material info"           : test_material,
                                "Key result"                   : gv(entry, "KeyResult"),
                                "Species / strain"             : gv(entry, "Organism"),
                                "Metabolic activation"         : gv(entry, "MetActIndicator"),
                                "Genotoxicity"                 : gv(entry, "Genotoxicity"),
                                "Cytotoxicity / top conc"      : gv(entry, "Cytotoxicity"),
                                "Vehicle controls validity"    : gv(entry, "VehContrValid"),
                                "Untreated negative ctrl valid": gv(entry, "NegContrValid"),
                                "True negative ctrl valid"     : None,
                                "Positive controls validity"   : gv(entry, "PosContrValid"),
                            })

    # ── 전체 i6z 탐색 ────────────────────────────────────────
    for dirpath, _, filenames in os.walk(root_dir):
        for filename in filenames:
            if not filename.endswith(".i6z"):
                continue

            file_path = os.path.join(dirpath, filename)

            try:
                z_outer = zipfile.ZipFile(file_path, 'r')
            except Exception:
                continue

            with z_outer:
                process_zip(z_outer, filename)

                for inner_name in [n for n in z_outer.namelist() if n.endswith(".i6z")]:
                    try:
                        with z_outer.open(inner_name) as inner_f:
                            inner_zip = zipfile.ZipFile(BytesIO(inner_f.read()), 'r')
                        with inner_zip:
                            process_zip(inner_zip, filename)
                    except Exception:
                        continue

    # ── CSV 저장 ─────────────────────────────────────────────
    fieldnames = [
        "파일명", "Compound name (IUPAC)", "CAS number", "SMILES",
        "Qualifier", "Guideline", "Deviation",
        "GLP compliance", "Type of assay", "Test material info",
        "Key result", "Species / strain", "Metabolic activation",
        "Genotoxicity", "Cytotoxicity / top conc",
        "Vehicle controls validity", "Untreated negative ctrl valid",
        "True negative ctrl valid", "Positive controls validity",
    ]

    with open(output_csv, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

    print(f"저장 완료: {output_csv}  ({len(rows)}행)")
    return rows


# ▶ 실행
rows = parse_all_genotox_to_csv(
    r"data1/reach_study_results_dossiers_23-05-2023",
    output_csv="genotox_full.csv"
)

저장 완료: genotox_full.csv  (142817행)


In [2]:
import pandas as pd

df = pd.read_csv("genotox_full.csv", dtype=str)

print(f"처리 전: {len(df)}행")

df = df[df["Genotoxicity"].notna()]

print(f"처리 후: {len(df)}행")
print(f"삭제된 행: {len(df) - len(df)}행")  

df.to_csv("genotox_full.csv", index=False, encoding="utf-8-sig")
print("저장 완료")

처리 전: 142817행
처리 후: 128388행
삭제된 행: 0행
저장 완료


In [3]:
# 파일 로드
df = pd.read_csv("genotox_full.csv", dtype=str)
phrase_df = pd.read_csv("dossier_test/list of fields.csv", dtype=str)

print(f"genotox_full.csv: {len(df)}행")
print(f"list of fields.csv: {len(phrase_df)}행")
print(f"list of fields 컬럼: {phrase_df.columns.tolist()}")

genotox_full.csv: 128388행
list of fields.csv: 78763행
list of fields 컬럼: ['PHRASEGROUP IDENTIFIER', 'PHRASE IDENTIFIER', 'PHRASE TEXT', 'PHRASE ADDITIONAL TEXT', 'OPEN', 'OBSOLETE', 'PARAMETERIZED', 'NOT-SELECTABLE', 'PHRASEGROUP SECURITY GROUP']


In [4]:
import pandas as pd

# 파일 로드
df = pd.read_csv("genotox_full.csv", dtype=str)
phrase_df = pd.read_csv("dossier_test/list of fields.csv", dtype=str)

# 매핑 딕셔너리 생성 (PHRASE IDENTIFIER → PHRASE TEXT)
phrase_map = dict(zip(
    phrase_df["PHRASE IDENTIFIER"].str.strip(),
    phrase_df["PHRASE TEXT"].str.strip()
))

# 매핑 적용할 컬럼들
cols_to_map = [
    "Qualifier", "Guideline", "Deviation",
    "GLP compliance", "Type of assay",
    "Species / strain", "Metabolic activation",
    "Genotoxicity", "Cytotoxicity / top conc",
    "Vehicle controls validity", "Untreated negative ctrl valid",
    "True negative ctrl valid", "Positive controls validity"
]

# Guideline은 | 로 여러 값이 합쳐져 있으므로 별도 처리
pipe_cols = ["Qualifier", "Guideline", "Deviation"]  # | 로 열거된 컬럼들
single_cols = [c for c in cols_to_map if c not in pipe_cols]

# | 로 열거된 컬럼 매핑
for col in pipe_cols:
    def map_pipe(val):
        if pd.isna(val) or str(val).strip() == "":
            return val
        parts = [v.strip() for v in str(val).split("|")]
        mapped = [phrase_map.get(p, p) for p in parts]
        return " | ".join(mapped)
    df[col] = df[col].apply(map_pipe)

# 단일 값 컬럼 매핑
for col in single_cols:
    df[col] = df[col].str.strip().map(phrase_map).fillna(df[col])

# 저장
df.to_csv("genotox_full_mapped.csv", index=False, encoding="utf-8-sig")
print(f"저장 완료: genotox_full_mapped.csv ({len(df)}행)")

# 샘플 확인
print("\n--- 매핑 결과 샘플 ---")
print(df[["Qualifier", "Guideline", "Genotoxicity", "Species / strain"]].head(5).to_string())

저장 완료: genotox_full_mapped.csv (128388행)

--- 매핑 결과 샘플 ---
                            Qualifier                                                                                             Guideline Genotoxicity   Species / strain
0  equivalent or similar to guideline                                                 OECD Guideline 471 (Bacterial Reverse Mutation Assay)     negative                NaN
1                                 NaN                                                                                                   NaN     negative                NaN
2  equivalent or similar to guideline                                                 OECD Guideline 471 (Bacterial Reverse Mutation Assay)     negative             other:
3  equivalent or similar to guideline                                                 OECD Guideline 471 (Bacterial Reverse Mutation Assay)     negative  E. coli WP2 uvr A
4  equivalent or similar to guideline  OECD Guideline 479 (Genetic Toxicology: In

In [5]:
# 'Compound name (IUPAC)', 'CAS number','SMILES'가 모두 null인 샘플은 삭제

df = pd.read_csv("genotox_full_mapped.csv", dtype=str)
print(f"처리 전: {len(df)}행")

mask = df[["Compound name (IUPAC)", "CAS number", "SMILES"]].notna().any(axis=1)
df = df[mask]

print(f"처리 후: {len(df)}행")
print(f"삭제된 행: {~mask.sum()}행")

df.to_csv("genotox_full_mapped.csv", index=False, encoding="utf-8-sig")
print("저장 완료")

처리 전: 128388행
처리 후: 126476행
삭제된 행: -126477행
저장 완료
